In [ ]:
#Import thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
#Load dữ liệu
df = pd.read_csv('/content/Data.csv')
df = df.dropna()

print(df.head())
print(df.columns)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Data.csv'

In [ ]:
# Chọn các cột số để vẽ
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Vẽ histogram cho từng cột
df[numeric_cols].hist(figsize=(10, 8))

plt.suptitle("Phân bố dữ liệu các biến")
plt.tight_layout()
plt.show()

In [ ]:
# Cập nhật lại mốc chia để cân bằng dữ liệu
def classify_growth(x):
    if x < 10.61:       # Sửa lại mức Thấp
        return 0
    elif x < 10.63:     # Sửa lại mức Trung Bình
        return 1
    else:               # Mức Cao
        return 2

df['Growth_Label'] = df['LGDP'].apply(classify_growth)

# Thêm dòng này để kiểm tra xem dữ liệu đã được chia đều chưa
print(df['Growth_Label'].value_counts())

Growth_Label
0    2386
2     326
1      17
Name: count, dtype: int64


In [ ]:
# Chọn features
features = [
    'IU: Internet use ',
    'BBS: Broadband Subscription',
    'GFCF: Gross Fixed Capital Formation ',
    'LCPI: Consumers Price Index '
]

for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna()
X = df[features].values.astype(float)
y = df['Growth_Label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
#Định nghĩa Node
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None

In [ ]:
#Decision Tree
class MyDecisionTree:
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        if depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split:
            return Node(value=self._most_common_label(y))

        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)

        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        if best_feat is None:
            return Node(value=self._most_common_label(y))

        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)

        left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)

        return Node(best_feat, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thr

        return split_idx, split_thresh

    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)
        left_idxs, right_idxs = self._split(X_column, threshold)

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)

        e_l = self._entropy(y[left_idxs])
        e_r = self._entropy(y[right_idxs])

        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy

    def _split(self, X_column, threshold):
        left_idxs = np.argwhere(X_column <= threshold).flatten()
        right_idxs = np.argwhere(X_column > threshold).flatten()
        return left_idxs, right_idxs

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [ ]:
#Random Forest
class MyRandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []

    def fit(self, X, y):
        self.trees = []

        for _ in range(self.n_trees):
            tree = MyDecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=self.n_features
            )
            X_samp, y_samp = self._bootstrap_sample(X, y)
            tree.fit(X_samp, y_samp)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)

        return np.array([Counter(pred).most_common(1)[0][0] for pred in tree_preds])

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

In [ ]:
#Train & Evaluate
n_features = int(np.sqrt(X.shape[1]))

tree = MyDecisionTree(max_depth=5, n_features=n_features)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)

rf = MyRandomForest(n_trees=20, max_depth=5, n_features=n_features)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, pred_tree))
print("Random Forest Accuracy:", accuracy_score(y_test, pred_rf))

Decision Tree Accuracy: 0.8754578754578755
Random Forest Accuracy: 0.891941391941392


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# 1. Tạo 4 thanh trượt (slider) cho 4 chỉ số kinh tế
iu_slider = widgets.FloatSlider(value=0.44, min=0.0, max=1.0, step=0.01, description='Internet (IU):')
bbs_slider = widgets.FloatSlider(value=0.02, min=0.0, max=1.0, step=0.01, description='Broadband:')
gfcf_slider = widgets.FloatSlider(value=0.23, min=0.0, max=1.0, step=0.01, description='Vốn (GFCF):')
lcpi_slider = widgets.FloatSlider(value=4.41, min=0.0, max=10.0, step=0.1, description='Lạm phát:')

# 2. Tạo nút bấm để dự báo
button = widgets.Button(description="Dự báo tăng trưởng!", button_style='success')
output = widgets.Output()

# 3. Định nghĩa hành động khi bấm nút
def on_button_clicked(b):
    with output:
        output.clear_output()
        # Gom số liệu từ 4 thanh trượt lại
        so_lieu = np.array([[iu_slider.value, bbs_slider.value, gfcf_slider.value, lcpi_slider.value]])

        # Đưa vào Rừng Ngẫu Nhiên (rf) để dự đoán
        muc_tang_truong = rf.predict(so_lieu)[0]

        # In kết quả
        print("-----------------------------------")
        if muc_tang_truong == 0:
            print("📉 KẾT QUẢ: TĂNG TRƯỞNG THẤP (0)")
        elif muc_tang_truong == 1:
            print("⚖️ KẾT QUẢ: TĂNG TRƯỞNG TRUNG BÌNH (1)")
        else:
            print("🚀 KẾT QUẢ: TĂNG TRƯỞNG CAO (2)")
        print("-----------------------------------")

button.on_click(on_button_clicked)

print("KÉO THANH TRƯỢT ĐỂ THAY ĐỔI CÁC CHỈ SỐ KINH TẾ:")
display(iu_slider, bbs_slider, gfcf_slider, lcpi_slider, button, output)

KÉO THANH TRƯỢT ĐỂ THAY ĐỔI CÁC CHỈ SỐ KINH TẾ:


FloatSlider(value=0.44, description='Internet (IU):', max=1.0, step=0.01)

FloatSlider(value=0.02, description='Broadband:', max=1.0, step=0.01)

FloatSlider(value=0.23, description='Vốn (GFCF):', max=1.0, step=0.01)

FloatSlider(value=4.41, description='Lạm phát:', max=10.0)

Button(button_style='success', description='Dự báo tăng trưởng!', style=ButtonStyle())

Output()